In [ ]:

import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import ConfusionMatrixDisplay

# 1. Load dataset
data = pd.read_csv('NASA Turbofan Engine.csv')

X = data.drop(['unit', 'cycle', 'RUL', 'Fault_Label'], axis=1)
y = data['Fault_Label']

# 2. Preprocess data
X_train = X[y == 0]

X_test = X

# Normalize data
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Reshape data for LSTM
X_train = X_train.reshape(-1, 1, X_train.shape[1])
X_test = X_test.reshape(-1, 1, X_test.shape[1])

# 3. Build Autoencoder
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1, X_train.shape[2])),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.RepeatVector(1),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(X_train.shape[2])
    )
])

# 4. Compile model
model.compile(optimizer='adam', loss='mse')

# 5. Train model
history = model.fit(
    X_train, X_train,
    epochs=10,
    batch_size=32
)

# 6. Calculate reconstruction error
train_pred = model.predict(X_train)

train_error = np.mean(
    (X_train - train_pred)**2, axis=(1,2)
)

threshold = np.percentile(train_error, 95)

test_pred = model.predict(X_test)

test_error = np.mean(
    (X_test - test_pred)**2, axis=(1,2)
)

# 7. Detect anomalies
y_pred = (test_error > threshold).astype(int)
y_true = y.values.astype(int)

# 8. Evaluate and plot
print("Threshold:", threshold)

plt.plot(history.history['loss'])
plt.title('Autoencoder Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
plt.show()
